In [ ]:
import sys, pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [ ]:
with open("../data/data.pkl", "rb") as f:
    data = pickle.load(f)

X_train  = data["X_train"]
X_test   = data["X_test"]
y_train  = data["y_train"]
y_test   = data["y_test"]
FEATURES = data["FEATURES"]

## Linear SVM

Tuning uses `LinearSVC` (O(n×p), no Platt scaling overhead) inside `GridSearchCV`.
`LinearSVC` exposes `decision_function`, so `scoring='roc_auc'` works without probabilities.
The final model wraps `LinearSVC` in `CalibratedClassifierCV(cv=3)` to obtain
`predict_proba` for threshold analysis — this calibration runs once per feature set,
not once per grid point.

In [ ]:
def make_linear_svm_tuning_pipeline(C=1.0):
    """Fast pipeline for grid search — no probability estimates needed."""
    return ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote",  SMOTE(random_state=42, k_neighbors=5)),
        ("model",  LinearSVC(C=C, max_iter=2000, random_state=42)),
    ])

def make_linear_svm_pipeline(C=1.0):
    """Final pipeline with predict_proba via Platt calibration."""
    return ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote",  SMOTE(random_state=42, k_neighbors=5)),
        ("model",  CalibratedClassifierCV(LinearSVC(C=C, max_iter=2000, random_state=42), cv=3)),
    ])

param_grid_lin = {"model__C": [0.01, 0.1, 1, 10, 100]}

gs_lin = GridSearchCV(
    make_linear_svm_tuning_pipeline(),
    param_grid_lin,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1,
)
gs_lin.fit(X_train[FEATURES["C_full"]], y_train)

best_C_lin = gs_lin.best_params_["model__C"]
print(f"Best params (linear): {gs_lin.best_params_}")
print(f"Best CV ROC-AUC: {gs_lin.best_score_:.4f}")

### Psych Questionnaire Features Only (Linear SVM)

In [ ]:
pipe_a_lin = make_linear_svm_pipeline(C=best_C_lin)
pipe_a_lin.fit(X_train[FEATURES["A_psych"]], y_train)

In [ ]:
y_train_pred = pipe_a_lin.predict(X_train[FEATURES["A_psych"]])
y_test_pred  = pipe_a_lin.predict(X_test[FEATURES["A_psych"]])

print("Train Performance:")
print(classification_report(y_train, y_train_pred))
print("Test Performance:")
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Linear SVM - Psychological Features")
plt.show()

In [ ]:
y_prob_a_lin = pipe_a_lin.predict_proba(X_test[FEATURES["A_psych"]])[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_a_lin):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_a_lin):.4f}")

In [ ]:
thresholds = np.linspace(0, 1, 101)
f1s, precs, recs = [], [], []
for t in thresholds:
    y_pred_t = (y_prob_a_lin >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    precs.append(precision_score(y_test, y_pred_t, zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, zero_division=0))

best_t = thresholds[np.argmax(f1s)]
plt.figure()
plt.plot(thresholds, f1s, label="F1")
plt.plot(thresholds, precs, label="Precision")
plt.plot(thresholds, recs, label="Recall")
plt.axvline(best_t, color="gray", linestyle="--", label=f"Best t = {best_t:.2f}")
plt.xlabel("Threshold")
plt.title("Threshold Sweep - Linear SVM - Psychological Features")
plt.legend()
plt.show()
print(f"Best threshold (max F1): {best_t:.2f}  →  F1 = {max(f1s):.4f}")

In [ ]:
lin_coef_a = np.mean(
    [c.estimator.coef_[0] for c in pipe_a_lin.named_steps["model"].calibrated_classifiers_],
    axis=0,
)
coef_df = (
    pd.DataFrame({"Feature": FEATURES["A_psych"], "Coefficient": lin_coef_a})
    .assign(abs_coef=lambda d: d["Coefficient"].abs())
    .sort_values("abs_coef", ascending=True)
    .drop(columns="abs_coef")
)
colors = ["#d73027" if c > 0 else "#4575b4" for c in coef_df["Coefficient"]]
fig, ax = plt.subplots(figsize=(7, max(4, len(coef_df) * 0.28)))
ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Linear SVM Coefficients - Psychological Features")
plt.tight_layout()
plt.show()

### Gaming Features Only (Linear SVM)

In [ ]:
pipe_b_lin = make_linear_svm_pipeline(C=best_C_lin)
pipe_b_lin.fit(X_train[FEATURES["B_gaming"]], y_train)

In [ ]:
y_train_pred = pipe_b_lin.predict(X_train[FEATURES["B_gaming"]])
y_test_pred  = pipe_b_lin.predict(X_test[FEATURES["B_gaming"]])

print("Train Performance:")
print(classification_report(y_train, y_train_pred))
print("Test Performance:")
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Linear SVM - Gaming Features")
plt.show()

In [ ]:
y_prob_b_lin = pipe_b_lin.predict_proba(X_test[FEATURES["B_gaming"]])[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_b_lin):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_b_lin):.4f}")

In [ ]:
thresholds = np.linspace(0, 1, 101)
f1s, precs, recs = [], [], []
for t in thresholds:
    y_pred_t = (y_prob_b_lin >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    precs.append(precision_score(y_test, y_pred_t, zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, zero_division=0))

best_t = thresholds[np.argmax(f1s)]
plt.figure()
plt.plot(thresholds, f1s, label="F1")
plt.plot(thresholds, precs, label="Precision")
plt.plot(thresholds, recs, label="Recall")
plt.axvline(best_t, color="gray", linestyle="--", label=f"Best t = {best_t:.2f}")
plt.xlabel("Threshold")
plt.title("Threshold Sweep - Linear SVM - Gaming Features")
plt.legend()
plt.show()
print(f"Best threshold (max F1): {best_t:.2f}  →  F1 = {max(f1s):.4f}")

In [ ]:
lin_coef_b = np.mean(
    [c.estimator.coef_[0] for c in pipe_b_lin.named_steps["model"].calibrated_classifiers_],
    axis=0,
)
coef_df = (
    pd.DataFrame({"Feature": FEATURES["B_gaming"], "Coefficient": lin_coef_b})
    .assign(abs_coef=lambda d: d["Coefficient"].abs())
    .sort_values("abs_coef", ascending=True)
    .drop(columns="abs_coef")
)
colors = ["#d73027" if c > 0 else "#4575b4" for c in coef_df["Coefficient"]]
fig, ax = plt.subplots(figsize=(7, max(4, len(coef_df) * 0.28)))
ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Linear SVM Coefficients - Gaming Features")
plt.tight_layout()
plt.show()

### Combined Gaming and Psych Features (Linear SVM)

In [ ]:
pipe_c_lin = make_linear_svm_pipeline(C=best_C_lin)
pipe_c_lin.fit(X_train[FEATURES["C_full"]], y_train)

In [ ]:
y_train_pred = pipe_c_lin.predict(X_train[FEATURES["C_full"]])
y_test_pred  = pipe_c_lin.predict(X_test[FEATURES["C_full"]])

print("Train Performance:")
print(classification_report(y_train, y_train_pred))
print("Test Performance:")
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Linear SVM - Psychological and Gaming Features")
plt.show()

In [ ]:
y_prob_c_lin = pipe_c_lin.predict_proba(X_test[FEATURES["C_full"]])[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_c_lin):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_c_lin):.4f}")

In [ ]:
thresholds = np.linspace(0, 1, 101)
f1s, precs, recs = [], [], []
for t in thresholds:
    y_pred_t = (y_prob_c_lin >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    precs.append(precision_score(y_test, y_pred_t, zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, zero_division=0))

best_t = thresholds[np.argmax(f1s)]
plt.figure()
plt.plot(thresholds, f1s, label="F1")
plt.plot(thresholds, precs, label="Precision")
plt.plot(thresholds, recs, label="Recall")
plt.axvline(best_t, color="gray", linestyle="--", label=f"Best t = {best_t:.2f}")
plt.xlabel("Threshold")
plt.title("Threshold Sweep - Linear SVM - Psychological and Gaming Features")
plt.legend()
plt.show()
print(f"Best threshold (max F1): {best_t:.2f}  →  F1 = {max(f1s):.4f}")

In [ ]:
lin_coef_c = np.mean(
    [c.estimator.coef_[0] for c in pipe_c_lin.named_steps["model"].calibrated_classifiers_],
    axis=0,
)
coef_df = (
    pd.DataFrame({"Feature": FEATURES["C_full"], "Coefficient": lin_coef_c})
    .assign(abs_coef=lambda d: d["Coefficient"].abs())
    .sort_values("abs_coef", ascending=True)
    .drop(columns="abs_coef")
)
colors = ["#d73027" if c > 0 else "#4575b4" for c in coef_df["Coefficient"]]
fig, ax = plt.subplots(figsize=(7, max(4, len(coef_df) * 0.28)))
ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Linear SVM Coefficients - Psychological and Gaming Features")
plt.tight_layout()
plt.show()

## RBF SVM

Tuning uses `SVC(probability=False)` in `GridSearchCV` — dropping Platt scaling during
the grid search avoids an internal CV per candidate. `gamma` is fixed to `'scale'`
(optimal for standardised data); only `C` is swept, giving 3 fits × 3 folds = 9 total.
The final model re-enables `probability=True` for `predict_proba`.

In [ ]:
def make_rbf_svm_tuning_pipeline(C=1.0):
    """Fast pipeline for grid search — probability=False avoids Platt scaling overhead."""
    return ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote",  SMOTE(random_state=42, k_neighbors=5)),
        ("model",  SVC(kernel="rbf", C=C, gamma="scale", random_state=42)),
    ])

def make_rbf_svm_pipeline(C=1.0):
    """Final pipeline with predict_proba."""
    return ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote",  SMOTE(random_state=42, k_neighbors=5)),
        ("model",  SVC(kernel="rbf", C=C, gamma="scale", probability=True, random_state=42)),
    ])

param_grid_rbf = {"model__C": [0.1, 1, 10]}

gs_rbf = GridSearchCV(
    make_rbf_svm_tuning_pipeline(),
    param_grid_rbf,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1,
)
gs_rbf.fit(X_train[FEATURES["C_full"]], y_train)

best_C_rbf = gs_rbf.best_params_["model__C"]
print(f"Best params (rbf): {gs_rbf.best_params_}")
print(f"Best CV ROC-AUC: {gs_rbf.best_score_:.4f}")

### Psych Questionnaire Features Only (RBF SVM)

In [ ]:
pipe_a_rbf = make_rbf_svm_pipeline(C=best_C_rbf)
pipe_a_rbf.fit(X_train[FEATURES["A_psych"]], y_train)

In [ ]:
y_train_pred = pipe_a_rbf.predict(X_train[FEATURES["A_psych"]])
y_test_pred  = pipe_a_rbf.predict(X_test[FEATURES["A_psych"]])

print("Train Performance:")
print(classification_report(y_train, y_train_pred))
print("Test Performance:")
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - RBF SVM - Psychological Features")
plt.show()

In [ ]:
y_prob_a_rbf = pipe_a_rbf.predict_proba(X_test[FEATURES["A_psych"]])[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_a_rbf):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_a_rbf):.4f}")

In [ ]:
thresholds = np.linspace(0, 1, 101)
f1s, precs, recs = [], [], []
for t in thresholds:
    y_pred_t = (y_prob_a_rbf >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    precs.append(precision_score(y_test, y_pred_t, zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, zero_division=0))

best_t = thresholds[np.argmax(f1s)]
plt.figure()
plt.plot(thresholds, f1s, label="F1")
plt.plot(thresholds, precs, label="Precision")
plt.plot(thresholds, recs, label="Recall")
plt.axvline(best_t, color="gray", linestyle="--", label=f"Best t = {best_t:.2f}")
plt.xlabel("Threshold")
plt.title("Threshold Sweep - RBF SVM - Psychological Features")
plt.legend()
plt.show()
print(f"Best threshold (max F1): {best_t:.2f}  →  F1 = {max(f1s):.4f}")

### Gaming Features Only (RBF SVM)

In [ ]:
pipe_b_rbf = make_rbf_svm_pipeline(C=best_C_rbf)
pipe_b_rbf.fit(X_train[FEATURES["B_gaming"]], y_train)

In [ ]:
y_train_pred = pipe_b_rbf.predict(X_train[FEATURES["B_gaming"]])
y_test_pred  = pipe_b_rbf.predict(X_test[FEATURES["B_gaming"]])

print("Train Performance:")
print(classification_report(y_train, y_train_pred))
print("Test Performance:")
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - RBF SVM - Gaming Features")
plt.show()

In [ ]:
y_prob_b_rbf = pipe_b_rbf.predict_proba(X_test[FEATURES["B_gaming"]])[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_b_rbf):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_b_rbf):.4f}")

In [ ]:
thresholds = np.linspace(0, 1, 101)
f1s, precs, recs = [], [], []
for t in thresholds:
    y_pred_t = (y_prob_b_rbf >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    precs.append(precision_score(y_test, y_pred_t, zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, zero_division=0))

best_t = thresholds[np.argmax(f1s)]
plt.figure()
plt.plot(thresholds, f1s, label="F1")
plt.plot(thresholds, precs, label="Precision")
plt.plot(thresholds, recs, label="Recall")
plt.axvline(best_t, color="gray", linestyle="--", label=f"Best t = {best_t:.2f}")
plt.xlabel("Threshold")
plt.title("Threshold Sweep - RBF SVM - Gaming Features")
plt.legend()
plt.show()
print(f"Best threshold (max F1): {best_t:.2f}  →  F1 = {max(f1s):.4f}")

### Combined Gaming and Psych Features (RBF SVM)

In [ ]:
pipe_c_rbf = make_rbf_svm_pipeline(C=best_C_rbf)
pipe_c_rbf.fit(X_train[FEATURES["C_full"]], y_train)

In [ ]:
y_train_pred = pipe_c_rbf.predict(X_train[FEATURES["C_full"]])
y_test_pred  = pipe_c_rbf.predict(X_test[FEATURES["C_full"]])

print("Train Performance:")
print(classification_report(y_train, y_train_pred))
print("Test Performance:")
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - RBF SVM - Psychological and Gaming Features")
plt.show()

In [ ]:
y_prob_c_rbf = pipe_c_rbf.predict_proba(X_test[FEATURES["C_full"]])[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_c_rbf):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob_c_rbf):.4f}")

In [ ]:
thresholds = np.linspace(0, 1, 101)
f1s, precs, recs = [], [], []
for t in thresholds:
    y_pred_t = (y_prob_c_rbf >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    precs.append(precision_score(y_test, y_pred_t, zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, zero_division=0))

best_t = thresholds[np.argmax(f1s)]
plt.figure()
plt.plot(thresholds, f1s, label="F1")
plt.plot(thresholds, precs, label="Precision")
plt.plot(thresholds, recs, label="Recall")
plt.axvline(best_t, color="gray", linestyle="--", label=f"Best t = {best_t:.2f}")
plt.xlabel("Threshold")
plt.title("Threshold Sweep - RBF SVM - Psychological and Gaming Features")
plt.legend()
plt.show()
print(f"Best threshold (max F1): {best_t:.2f}  →  F1 = {max(f1s):.4f}")